loading the data and prepare the libraries

In [22]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [14]:
train_path = Path.cwd() / "Data" /"train.csv"

train_data = pd.read_csv(train_path)

train_data.head()

,recording,day,weekday,unit,minute,minute_of_day,people
0,20260904_1105,2026-09-04,Fri,cam_B,11:16,676,13.103
1,20260904_1105,2026-09-04,Fri,cam_B,11:17,677,17.157
2,20260904_1105,2026-09-04,Fri,cam_B,11:18,678,23.348
3,20260904_1105,2026-09-04,Fri,cam_B,11:19,679,15.478
4,20260904_1105,2026-09-04,Fri,cam_B,11:20,680,16.407


EDA

In [15]:
train_data.info() # inspect the data typer and 

<class 'pandas.DataFrame'>
RangeIndex: 13443 entries, 0 to 13442
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   recording      13443 non-null  str    
 1   day            13443 non-null  str    
 2   weekday        13443 non-null  str    
 3   unit           13443 non-null  str    
 4   minute         13443 non-null  str    
 5   minute_of_day  13443 non-null  int64  
 6   people         13443 non-null  float64
dtypes: float64(1), int64(1), str(5)
memory usage: 735.3 KB


In [16]:
train_data.isnull().sum() # checking for the null values 

recording        0
day              0
weekday          0
unit             0
minute           0
minute_of_day    0
people           0
dtype: int64

In [17]:
train_data.describe()

,minute_of_day,people
count,13443.000000,13443.000000
mean,908.839768,81.575230
std,243.275264,111.203073
min,266.000000,1.119000
25%,753.000000,12.322000
50%,903.000000,28.575000
75%,1122.000000,120.333000
max,1304.000000,998.501000


Feature Engineering

In [25]:

df = train_data.copy()

# Convert date and time columns
df["day"] = pd.to_datetime(df["day"])
df["minute"] = pd.to_datetime(df["minute"], format="%H:%M")

# Extract time features
df["hour"] = df["minute"].dt.hour
df["minute_component"] = df["minute"].dt.minute
df["day_of_month"] = df["day"].dt.day
df["month"] = df["day"].dt.month
df["day_of_week"] = df["day"].dt.dayofweek
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

# Cyclical time features
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

df["minute_sin"] = np.sin(2 * np.pi * df["minute_of_day"] / 1440)
df["minute_cos"] = np.cos(2 * np.pi * df["minute_of_day"] / 1440)

# Sort chronologically
df = df.sort_values(["unit", "recording", "minute_of_day"])

# Historical features
group = df.groupby(["unit", "recording"])["people"]

for lag in [1, 2, 5, 10, 15, 30, 60]:
    df[f"people_lag_{lag}"] = group.shift(lag)

# Rolling features: shift first to prevent data leakage
for window in [5, 15, 30]:
    df[f"people_rolling_mean_{window}"] = (
        group.transform(
            lambda x: x.shift(1).rolling(window).mean()
        )
    )

    df[f"people_rolling_std_{window}"] = (
        group.transform(
            lambda x: x.shift(1).rolling(window).std()
        )
    )

# Changes in people count
df["people_diff_1"] = group.diff(1)
df["people_diff_5"] = group.diff(5)

# Target: people count in the next minute
df["target"] = group.shift(-1)

# Remove rows without sufficient history or a future target
df = df.dropna().reset_index(drop=True)

df.head()

C:\Users\DELL\AppData\Local\Temp\ipykernel_25872\621859655.py:12: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df["day_of_week"] = df["day"].dt.dayofweek


,recording,day,weekday,unit,minute,minute_of_day,people,hour,minute_component,day_of_month,...,people_lag_60,people_rolling_mean_5,people_rolling_std_5,people_rolling_mean_15,people_rolling_std_15,people_rolling_mean_30,people_rolling_std_30,people_diff_1,people_diff_5,target
0,20260904_1105,2026-09-04,Fri,cam_A,1900-01-01 12:05:00,725,53.537,12,5,4,...,259.599,58.5314,4.913663,60.436333,10.489792,81.170667,50.083401,0.047,-12.723,61.029
1,20260904_1105,2026-09-04,Fri,cam_A,1900-01-01 12:06:00,726,61.029,12,6,4,...,264.964,55.9868,2.711742,60.319400,10.560782,73.753133,34.176050,7.492,2.810,75.211
2,20260904_1105,2026-09-04,Fri,cam_A,1900-01-01 12:07:00,727,75.211,12,7,4,...,275.221,56.5488,3.474047,60.546400,10.535271,68.351100,19.348896,14.182,15.804,78.448
3,20260904_1105,2026-09-04,Fri,cam_A,1900-01-01 12:08:00,728,78.448,12,8,4,...,262.843,59.7096,9.198249,61.452533,11.197802,66.220767,14.092727,3.237,23.167,61.752
4,20260904_1105,2026-09-04,Fri,cam_A,1900-01-01 12:09:00,729,61.752,12,9,4,...,257.342,64.3430,11.859640,62.899800,11.924407,65.415333,12.546930,-16.696,8.262,71.925


In [27]:
# Date features
df["year"] = df["day"].dt.year
df["month"] = df["day"].dt.month
df["day_of_month"] = df["day"].dt.day
df["day_of_week"] = df["day"].dt.dayofweek

# Time features
df["hour"] = df["minute"].dt.hour
df["minute_component"] = df["minute"].dt.minute

C:\Users\DELL\AppData\Local\Temp\ipykernel_25872\3243543801.py:5: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df["day_of_week"] = df["day"].dt.dayofweek


In [28]:
df.drop(columns=["day", "minute" , "weekday" , "recording" , "minute_of_day"], inplace=True)
df.head()

C:\Users\DELL\AppData\Local\Temp\ipykernel_25872\3586746950.py:1: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(columns=["day", "minute" , "weekday" , "recording" , "minute_of_day"], inplace=True)


,unit,people,hour,minute_component,day_of_month,month,day_of_week,is_weekend,hour_sin,hour_cos,...,people_rolling_mean_5,people_rolling_std_5,people_rolling_mean_15,people_rolling_std_15,people_rolling_mean_30,people_rolling_std_30,people_diff_1,people_diff_5,target,year
0,cam_A,53.537,12,5,4,9,4,0,1.224647e-16,-1.0,...,58.5314,4.913663,60.436333,10.489792,81.170667,50.083401,0.047,-12.723,61.029,2026
1,cam_A,61.029,12,6,4,9,4,0,1.224647e-16,-1.0,...,55.9868,2.711742,60.319400,10.560782,73.753133,34.176050,7.492,2.810,75.211,2026
2,cam_A,75.211,12,7,4,9,4,0,1.224647e-16,-1.0,...,56.5488,3.474047,60.546400,10.535271,68.351100,19.348896,14.182,15.804,78.448,2026
3,cam_A,78.448,12,8,4,9,4,0,1.224647e-16,-1.0,...,59.7096,9.198249,61.452533,11.197802,66.220767,14.092727,3.237,23.167,61.752,2026
4,cam_A,61.752,12,9,4,9,4,0,1.224647e-16,-1.0,...,64.3430,11.859640,62.899800,11.924407,65.415333,12.546930,-16.696,8.262,71.925,2026


In [ ]:
# # prepare the data for preprocessing and feature engineering
# numerical_features = train_data.select_dtypes(include=['int64', 'float64']).columns.tolist()

# categorical_features = train_data.select_dtypes(include=['str']).columns.tolist()

# print("Numerical Features:", numerical_features)
# print("Categorical Features:", categorical_features)

Numerical Features: ['minute_of_day', 'people']
Categorical Features: ['recording', 'day', 'weekday', 'unit', 'minute']
